# Demo: Proof Reconstructor

This notebook demonstrates the current explanation-reconstruction slice in `rdflib-reasoning-engine`.
It rebuilds a `DirectProof` from engine-native `DerivationRecord` values using `DerivationProofReconstructor`.

Current scope:
- triple goals only
- recursive reconstruction from derivation logs
- `RuleApplication` nodes for explained conclusions
- `ProofLeaf` fallback for unexplained premises or goals

This notebook is also the place to compare proof renderings and inspect the raw proof payload.

In [1]:
from pprint import pprint

from rdflib import BNode, Graph, URIRef
from rdflib.namespace import RDF, RDFS
from rdflib_reasoning.engine import (
    DerivationProofReconstructor,
    DerivationRecord,
    RuleId,
    TripleFact,
)

In [2]:
context = BNode()
display_graph = Graph()
display_graph.bind("ex", "urn:test:")

alice = URIRef("urn:test:alice")
human = URIRef("urn:test:Human")
mammal = URIRef("urn:test:Mammal")
animal = URIRef("urn:test:Animal")

goal = TripleFact(context=context, triple=(alice, RDF.type, animal))
intermediate = TripleFact(context=context, triple=(alice, RDF.type, mammal))
premise_a = TripleFact(context=context, triple=(alice, RDF.type, human))
premise_b = TripleFact(context=context, triple=(human, RDFS.subClassOf, mammal))
premise_c = TripleFact(context=context, triple=(mammal, RDFS.subClassOf, animal))

records = [
    DerivationRecord(
        context=context,
        conclusions=[intermediate],
        premises=[premise_a, premise_b],
        rule_id=RuleId(ruleset="rdfs", rule_id="rdfs9"),
        depth=1,
    ),
    DerivationRecord(
        context=context,
        conclusions=[goal],
        premises=[intermediate, premise_c],
        rule_id=RuleId(ruleset="rdfs", rule_id="rdfs9"),
        depth=2,
    ),
]

proof = DerivationProofReconstructor().reconstruct(goal, records)
proof

DirectProof(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), goal=TripleFact(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Animal'))), proof=RuleApplication(node_kind='rule_application', conclusions=[TripleFact(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Animal')))], premises=[RuleApplication(node_kind='rule_application', conclusions=[TripleFact(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), kind='triple', triple=(rdflib.term.URIRef('urn:test:alice'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Mammal')))], premises=[ProofLeaf(node_kind='leaf', c

## 1. Render the Proof as a Diagram

Mermaid is useful when you want to see the proof structure and rule nesting at a glance.

In [3]:
from rdflib_reasoning.engine.proof_notebook import display_proof_mermaid

display_proof_mermaid(proof, namespace_manager=display_graph.namespace_manager)

```mermaid
flowchart TD
n2[["rdfs:rdfs9
IF (?x, rdfs:subClassOf, ?y) AND (?a, rdf:type, ?x) AND different_terms(?x, ?y)
THEN (?a, rdf:type, ?y)"]]
n3("(ex:alice rdf:type ex:Animal)")
n3 -->|entailed_by| n2
n4[["rdfs:rdfs9
IF (?x, rdfs:subClassOf, ?y) AND (?a, rdf:type, ?x) AND different_terms(?x, ?y)
THEN (?a, rdf:type, ?y)"]]
n5("(ex:alice rdf:type ex:Mammal)")
n5 -->|entailed_by| n4
n6["(ex:alice rdf:type ex:Human)"]
n4 -->|had_premise| n6
n7["(ex:Human rdfs:subClassOf ex:Mammal)"]
n4 -->|had_premise| n7
n2 -->|had_premise| n5
n8["(ex:Mammal rdfs:subClassOf ex:Animal)"]
n2 -->|had_premise| n8
n1>"(ex:alice rdf:type ex:Animal)"]
n1 -->|justified_by| n3
```

## 2. Render the Proof as Markdown

The markdown view is a better fit when you want a compact text explanation instead of a graph visualization.

In [4]:
from rdflib_reasoning.engine.proof_notebook import display_proof_markdown

display_proof_markdown(proof, namespace_manager=display_graph.namespace_manager)

## Direct Proof

- Context: `_:Nb7d1cc6de98a41d691597cb2b41cc883`
- Goal: `(ex:alice rdf:type ex:Animal)`
- Verdict: `proved`

### Steps

- Rule Application: `rdfs:rdfs9`
  - Conclusions:
    - `(ex:alice rdf:type ex:Animal)`
  - Premises:
    - Rule Application: `rdfs:rdfs9`
      - Conclusions:
        - `(ex:alice rdf:type ex:Mammal)`
      - Premises:
        - Leaf: `(ex:alice rdf:type ex:Human)`
        - Leaf: `(ex:Human rdfs:subClassOf ex:Mammal)`
    - Leaf: `(ex:Mammal rdfs:subClassOf ex:Animal)`

## 3. Inspect the Proof Object

The reconstructed proof remains a typed data model, so you can inspect its structure directly or serialize it for other tooling.

In [5]:
print("Verdict:", proof.verdict)
print("Top-level node kind:", proof.proof.node_kind)
print(
    "Nested premise node kinds:",
    [premise.node_kind for premise in proof.proof.premises],
)

pprint(proof.model_dump(mode="python"))

Verdict: proved
Top-level node kind: rule_application
Nested premise node kinds: ['rule_application', 'leaf']
{'context': rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'),
 'goal': {'context': rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'),
          'kind': 'triple',
          'triple': (rdflib.term.URIRef('urn:test:alice'),
                     rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                     rdflib.term.URIRef('urn:test:Animal'))},
 'notes': None,
 'proof': {'conclusions': [{'context': rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'),
                            'kind': 'triple',
                            'triple': (rdflib.term.URIRef('urn:test:alice'),
                                       rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                                       rdflib.term.URIRef('urn:test:Animal'))}],
           'derivation': {'bindings': [],
                          'conclusion

## 4. Missing Derivations Fall Back to Leaves

In [6]:
# If no derivation record exists for the goal, reconstruction falls back to a leaf.
unexplained_goal = TripleFact(
    context=context,
    triple=(URIRef("urn:test:bob"), RDF.type, URIRef("urn:test:Human")),
)

unexplained = DerivationProofReconstructor().reconstruct(unexplained_goal, records)
print("Verdict:", unexplained.verdict)
print("Proof node kind:", unexplained.proof.node_kind)
unexplained

Verdict: incomplete
Proof node kind: leaf


DirectProof(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), goal=TripleFact(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), kind='triple', triple=(rdflib.term.URIRef('urn:test:bob'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Human'))), proof=ProofLeaf(node_kind='leaf', claim=TripleFact(context=rdflib.term.BNode('Nb7d1cc6de98a41d691597cb2b41cc883'), kind='triple', triple=(rdflib.term.URIRef('urn:test:bob'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('urn:test:Human'))), grounding=[]), verdict='incomplete', notes=None)